# Exercise 3 — Accept or reject the batch, in Python
*Quality Management · Hands-on lab · Università degli Studi di Perugia*

What the [web page](https://lollocappo.github.io/student-hub/materials/quality-management/lab/batch.html) does, with pandas. Run the cells in order with **Shift + Enter**.

In [ ]:
# Run this cell first: it downloads the example files from the course repository.
import os, urllib.request
BASE = "https://raw.githubusercontent.com/LolloCappo/student-hub/main/materials/quality-management/lab/"

def fetch(path):
    """Download a file of the lab once, and return its local name."""
    name = os.path.basename(path)
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + path, name)
    return name

FILES = [fetch("data/example-class.csv")]   # six groups, 15 washers each, 3 photos per washer

In [ ]:
# Optional — use your own class's CSV files (select them all) instead of the example.
# Remove the "#" in front of the two lines below, run the cell, and pick the file.
# from google.colab import files
# FILES = list(files.upload())

## 1. The data and the specification
One washer = group + part name. Masters are left out: their values were set by the caliper.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

df = pd.concat([pd.read_csv(f).assign(group=lambda d, f=f: d.get("group", pd.Series(dtype=str)).fillna(os.path.splitext(f)[0])) for f in FILES])
df = df[df.master.astype(str).str.lower() != "true"]

COL = "id_mm"                 # "od_mm" for the outer diameter d2, "id_mm" for the hole d1
LSL, USL = {"od_mm": (15.57, 16.00), "id_mm": (8.40, 8.62)}[COL]       # ISO 7089, M8, product grade A
U_CAL = 0.012                 # caliper standard uncertainty, mm (MPE 0.02 mm / sqrt(3))
by = df.groupby(["group", "part"])[COL]
print(f"{len(df)} measurements, {by.ngroups} washers, tolerance {LSL}–{USL} mm")

## 2. How good is the gauge?
Pooled repeatability from the repeated photos, then the standard uncertainty of a washer's mean value (GUM).

In [ ]:
n_i, var_i = by.count(), by.var()
s_r = np.sqrt((var_i * (n_i - 1)).sum() / (n_i - 1).sum())
n = n_i.median()
u = np.sqrt(s_r**2 / n + U_CAL**2)
ratio = 6 * u / (USL - LSL)
print(f"repeatability s_r = {s_r * 1000:.1f} µm per photo; mean of {n:.0f} photos -> {s_r / np.sqrt(n) * 1000:.1f} µm")
print(f"standard uncertainty u = {u * 1000:.1f} µm, expanded U = 2u = {2 * u * 1000:.0f} µm")
print(f"gauge-to-tolerance 6u/T = {ratio:.0%}  (AIAG guideline: <10 % good, 10–30 % marginal, >30 % not adequate)")
print(df.groupby("group").apply(lambda g: np.sqrt(g.groupby("part")[COL].var().mean()), include_groups=False).round(4).rename("s_r per group (mm)"))

## 3. The batch
A box has no production order, so this is process **performance** (P<sub>p</sub>, P<sub>pk</sub>, ISO 22514-2).

In [ ]:
x = by.mean()
m, s = x.mean(), x.std()
Pp, Ppk = (USL - LSL) / (6 * s), min(USL - m, m - LSL) / (3 * s)
ppm = 1e6 * (norm.cdf(LSL, m, s) + norm.sf(USL, m, s))
print(f"{len(x)} washers: mean {m:.3f} mm, s {s * 1000:.1f} µm, Pp {Pp:.2f}, Ppk {Ppk:.2f}, expected out of tolerance {ppm:.0f} ppm")
print(f"share of the batch variance that is the gauge: {(s_r**2 / n) / s**2:.0%}")

plt.figure(figsize=(9, 3.5))
plt.hist(x, bins=int(np.sqrt(len(x))), color="#2B4D9A", edgecolor="white")
v = np.linspace(min(x.min(), LSL) - 0.02, max(x.max(), USL) + 0.02, 300)
plt.plot(v, len(x) * (x.max() - x.min()) / int(np.sqrt(len(x))) * norm.pdf(v, m, s), "k--", label="normal model")
for lim, name in ((LSL, "LSL"), (USL, "USL")): plt.axvline(lim, color="k", lw=2); plt.text(lim, plt.ylim()[1], name, ha="center", va="bottom")
plt.xlabel(f"{COL} (mm)"); plt.ylabel("washers"); plt.legend(); plt.show()

## 4. Decide, washer by washer — ISO 14253-1
Default rule: conformity (or nonconformity) is proven only with at least 95 % probability, i.e. at least 1.645 u inside (or outside) the limits.

In [ ]:
g = 1.645 * u
iso = np.select([(x >= LSL + g) & (x <= USL - g), (x < LSL - g) | (x > USL + g)], ["conforming", "nonconforming"], "cannot decide")
simple = np.where((x >= LSL) & (x <= USL), "conforming", "nonconforming")
print(f"guard band g = {g * 1000:.0f} µm")
pd.DataFrame({"simple acceptance": pd.Series(simple).value_counts(), "ISO 14253-1 (95 %)": pd.Series(iso).value_counts()}).fillna(0).astype(int)

In [ ]:
groups = sorted(x.index.get_level_values("group").unique())
colour = {"conforming": "#1f7a3a", "cannot decide": "#a15c00", "nonconforming": "#b42318"}
marker = {"conforming": "o", "cannot decide": "D", "nonconforming": "X"}
plt.figure(figsize=(10, 0.5 * len(groups) + 1.2))
for lim in (LSL, USL):
    plt.axvspan(lim - g, lim + g, color="#fff3df"); plt.axvline(lim, color="k", lw=2)
for status in colour:
    sel = iso == status
    yy = [groups.index(gr) for gr in x.index.get_level_values("group")[sel]]
    plt.scatter(x.values[sel], yy, c=colour[status], marker=marker[status], s=40, label=status, edgecolors="white")
plt.yticks(range(len(groups)), groups); plt.xlabel(f"{COL} (mm)"); plt.legend(loc="upper left", fontsize=8); plt.show()

## Your decision
Accept, reject, or accept with conditions? Write one paragraph using P<sub>pk</sub>, the expected fraction out of tolerance, the decision counts and the gauge's own limits — then answer the questions on the web page.